# Multi-Agent Coordination — Demo Notebook

This notebook walks through training, evaluation, and visualisation of the
shared Double-DQN agent.  All implementation lives in the `src/` package;
this notebook just orchestrates calls and renders results.

Run order:
1. Imports and config
2. Train (≈ 4 minutes, or load a saved checkpoint)
3. Aggregate evaluation
4. Per-trial diagnostic table
5. Animated rollouts

## 1. Setup

In [1]:
import sys
from pathlib import Path

# Make `src` importable when running from notebooks/
sys.path.insert(0, str(Path.cwd().parent))

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')

import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import PillowWriter
from IPython.display import HTML, display

from src.env import EnvConfig
from src.agent import AgentConfig, DQNAgent
from src.train import TrainConfig, train
from src.evaluate import evaluate_round_trip, evaluate_detailed
from src.visualize import roll_out, animate_rollout

## 2. Load config and build the three config objects

All hyperparameters live in `configs/default.yaml` — change them there, not here.

In [2]:
with open('../configs/default.yaml') as f:
    cfg = yaml.safe_load(f)

env_cfg = EnvConfig(**cfg['env'],
                    use_opposite_sensor=cfg['options']['opposite_sensor'])
agent_cfg = AgentConfig(**cfg['agent'])
train_cfg = TrainConfig(
    total_step_cap=cfg['training']['total_step_cap'],
    collision_cap=cfg['training']['collision_cap'],
    walltime_cap=cfg['training']['walltime_cap'],
    layout_reset_every=cfg['training']['layout_reset_every'],
    mini_eval_every=cfg['training']['mini_eval_every'],
    early_stop_rate=cfg['training']['early_stop_rate'],
    seed=cfg['training']['seed'],
    central_clock=cfg['options']['central_clock'],
    phases=cfg['phases'],
)
print('configs ready')

configs ready


## 3. Train (or load)

Set `RETRAIN = False` to skip training and load the checkpoint from `../models/best.pt`.
Training takes about 4 minutes on CPU and runs the 3-phase curriculum with early stopping.

In [3]:
RETRAIN = False
CHECKPOINT = '../models/best.pt'

if RETRAIN:
    agent, history, train_collisions = train(env_cfg, agent_cfg, train_cfg)
    agent.save(CHECKPOINT)
    print(f'saved {CHECKPOINT} | collisions={train_collisions}')
else:
    agent = DQNAgent(agent_cfg)
    agent.load(CHECKPOINT)
    history, train_collisions = None, None
    print(f'loaded {CHECKPOINT}')

loaded ../models/best.pt


## 4. Aggregate evaluation

Each trial places agent 0 at B (empty) and asks it to complete a full B→A→B
round-trip.  The other 3 agents act greedily under the same shared policy.

In [4]:
print('=== Spec minimum-grade horizon (≤ 25 steps) ===')
s25 = evaluate_round_trip(agent, env_cfg, trials=cfg['evaluation']['trials'],
                          max_steps=25, central_clock=cfg['options']['central_clock'])
print()
print('=== Performance-point horizon (≤ 20 steps) ===')
s20 = evaluate_round_trip(agent, env_cfg, trials=cfg['evaluation']['trials'],
                          max_steps=20, central_clock=cfg['options']['central_clock'])

=== Spec minimum-grade horizon (≤ 25 steps) ===
EVAL (B→A→B, ≤25 steps): 94.40%  (472/500)   avg-steps-on-success=6.89   collisions=6

=== Performance-point horizon (≤ 20 steps) ===
EVAL (B→A→B, ≤20 steps): 94.40%  (472/500)   avg-steps-on-success=6.89   collisions=6


## 5. Per-trial diagnostic table

For inspection: each trial gets categorised as SUCCESS, COLLISION,
PICKED_UP_NOT_DELIVERED, or NEVER_REACHED_A.  The scenario seed is recorded
so the same trial can be replayed and animated below.

In [5]:
results = evaluate_detailed(agent, env_cfg, n_trials=20, max_steps=25, seed=2024,
                            central_clock=cfg['options']['central_clock'])
df = pd.DataFrame([r.__dict__ for r in results])

print('Outcomes:')
print(df['outcome'].value_counts().to_string())
print()
df

Outcomes:
outcome
SUCCESS                    19
PICKED_UP_NOT_DELIVERED     1



,trial,scenario_seed,A,B,optimal_steps,picked_up_at_step,delivered_at_step,collided_at_step,steps_taken,outcome
0,0,504756065,"(1, 3)","(0, 0)",8,4,8.0,None,8,SUCCESS
1,1,195105371,"(1, 2)","(3, 0)",8,4,8.0,None,8,SUCCESS
2,2,781967831,"(2, 1)","(3, 2)",4,2,4.0,None,4,SUCCESS
3,3,621001615,"(1, 2)","(3, 0)",8,4,8.0,None,8,SUCCESS
4,4,326150537,"(1, 1)","(3, 0)",6,3,6.0,None,6,SUCCESS
5,5,214964977,"(2, 3)","(1, 4)",4,2,4.0,None,4,SUCCESS
6,6,952729257,"(1, 1)","(3, 2)",6,3,8.0,None,8,SUCCESS
7,7,776868167,"(4, 1)","(2, 4)",10,5,10.0,None,10,SUCCESS
8,8,440329269,"(4, 3)","(0, 0)",14,7,14.0,None,14,SUCCESS
9,9,813055664,"(0, 2)","(1, 0)",6,3,NaN,None,25,PICKED_UP_NOT_DELIVERED


## 6. Inspect failures

If any trials failed, look at them here.

In [6]:
failures = df[df['outcome'] != 'SUCCESS']
if len(failures) == 0:
    print('No failures in this batch.')
else:
    print(f'{len(failures)} failure(s):')
    display(failures)

1 failure(s):


,trial,scenario_seed,A,B,optimal_steps,picked_up_at_step,delivered_at_step,collided_at_step,steps_taken,outcome
9,9,813055664,"(0, 2)","(1, 0)",6,3,NaN,None,25,PICKED_UP_NOT_DELIVERED


## 7. Animated rollouts

Three different seeds so you can see how the policy handles different A/B placements.

In [7]:
for seed in [7, 13, 42]:
    frames = roll_out(agent, env_cfg, max_steps=25, seed=seed,
                      central_clock=cfg['options']['central_clock'])
    print(f'seed={seed}: {len(frames)} micro-steps, '
          f'{sum(1 for f in frames if f["collided"])} collisions')
    anim = animate_rollout(frames, grid=env_cfg.grid,
                           title=f'Trained agents — seed={seed}', fps=3)
    display(HTML(anim.to_jshtml()))

2026-05-14 13:16:29,533 INFO Animation.save using <class 'matplotlib.animation.HTMLWriter'>


seed=7: 17 micro-steps, 0 collisions


2026-05-14 13:16:30,313 INFO Animation.save using <class 'matplotlib.animation.HTMLWriter'>


seed=13: 17 micro-steps, 0 collisions


2026-05-14 13:16:31,057 INFO Animation.save using <class 'matplotlib.animation.HTMLWriter'>


seed=42: 25 micro-steps, 0 collisions


## 8. Animate one specific trial

Use any row from the diagnostic table.  The scenario seed makes the rollout
deterministic — same A, same B, same other-agent placements, same actions.

In [8]:
def animate_trial(agent, env_cfg, scenario_seed, max_steps=25,
                  title_prefix='', central_clock=True):
    """Replay and animate a specific scenario."""
    frames = roll_out(agent, env_cfg, max_steps=max_steps,
                      seed=scenario_seed, central_clock=central_clock)
    anim = animate_rollout(frames, grid=env_cfg.grid,
                           title=f'{title_prefix} (seed={scenario_seed})', fps=3)
    return HTML(anim.to_jshtml())


# Pick a row from the diagnostic table — defaults to first failure, or trial 0.
if len(failures) > 0:
    row = failures.iloc[0]
    label = f'FAILURE [{row["outcome"]}]'
else:
    row = df.iloc[0]
    label = 'SUCCESS'

print(f"Animating trial {row['trial']}: {row['outcome']}  "
      f"A={row['A']} B={row['B']} steps={row['steps_taken']}/{row['optimal_steps']}")
animate_trial(agent, env_cfg, int(row['scenario_seed']), title_prefix=label,
              central_clock=cfg['options']['central_clock'])

2026-05-14 13:16:32,220 INFO Animation.save using <class 'matplotlib.animation.HTMLWriter'>


Animating trial 9: PICKED_UP_NOT_DELIVERED  A=(0, 2) B=(1, 0) steps=25/6
